# 🛡️ TransferGuard: Detecting and Avoiding Negative Transfer in Lightweight Vision Models

> **Computer Vision Course Project**  
> **Group Members:**  
> - **Ghanishth Saini** (Roll: 24121)  
> - **Arvind Yadav** (Roll: 24059)  
> - **Aakash Jagtap** (Roll: 24157)  

---

### 📌 Overview & Resumable Cloud Execution
This notebook executes the complete experimental benchmark on Google Colab GPU (T4/V100/A100).

✨ **Zero Progress Lost:** All training commands feature **automatic checkpoint resumption** (). If your Colab runtime disconnects or restarts midway through training, re-running the cells will automatically detect  and seamlessly continue from the exact epoch where it was interrupted!

**Hardware Acceleration Setup:**
Go to **Runtime > Change runtime type > Hardware accelerator > T4 GPU** before executing the cells below.

## 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")

## 2. (Optional) Mount Google Drive for Persistent Storage
Mounting Google Drive ensures that checkpoints and figures are permanently preserved in your Drive even if the Colab instance is destroyed.

In [ ]:
# Uncomment to mount Google Drive:
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/TransferGuard_Results"

## 3. Clone Repository & Install Dependencies

In [ ]:
# Clone repository from GitHub
!git clone https://github.com/arvindyadav186/TransferGuard-Detecting-and-Avoiding-Negative-Transfer-in-Lightweight-Vision-Models.git
%cd TransferGuard-Detecting-and-Avoiding-Negative-Transfer-in-Lightweight-Vision-Models
!pip install -q -r requirements.txt
!pip install -q -e .
print("Repository cloned and installed successfully!")

## 4. Run Automated Test Suite (Sanity Check)
Verifies all 11 unit tests (Linear CKA invariance, layer activation hooks, dynamic freezing, and resumable training).

In [ ]:
!pytest tests/ -v

## 5. Layer-wise CKA Representation Alignment Analysis
Computes Centered Kernel Alignment (CKA) between ImageNet pre-trained representations and target domain representations to detect the "divergence cliff".

In [ ]:
!python scripts/run_cka_analysis.py --dataset svhn --model mobilenet_v3_small --device auto --subset 500
from IPython.display import Image, display
display(Image("results/cka/cka_mobilenet_v3_small_svhn.png"))

## 6. Train Baselines & Proposed TransferGuard (With Auto-Resume)
All training commands feature . If interrupted, simply re-run the cell and training will continue from the latest epoch without starting over.

In [ ]:
# 1. Baseline: Train from Scratch (Random Initialization)
!python scripts/train_scratch.py --config configs/svhn_mobilenetv3.yaml --epochs 30 --device auto --resume auto

In [ ]:
# 2. Baseline: Standard Fine-Tuning (Demonstrating Negative Transfer)
!python scripts/train_finetune.py --config configs/svhn_mobilenetv3.yaml --mode full --epochs 25 --device auto --resume auto

In [ ]:
# 3. Baseline: Linear Probing (Frozen Backbone)
!python scripts/train_finetune.py --config configs/svhn_mobilenetv3.yaml --mode linear_probe --epochs 20 --device auto --resume auto

In [ ]:
# 4. Proposed Method: TransferGuard Adaptive Routing & Dynamic Freezing
!python scripts/train_transferguard.py --config configs/svhn_mobilenetv3.yaml --threshold 0.65 --epochs 25 --device auto --resume auto

## 7. Comprehensive Benchmark Evaluation & Transfer Gain Table
Computes Transfer Gain  = Acc_{transfer} - Acc_{scratch}$ and checks for negative transfer across all datasets.

In [ ]:
!python scripts/evaluate_all.py

## 8. Generate 300 DPI Publication & Presentation Figures

In [ ]:
!python scripts/plot_results.py --output-dir results/figures

from IPython.display import Image, display
print("--- Figure 1: Transfer Gain Comparison (Positive vs Negative Transfer Zones) ---")
display(Image("results/figures/transfer_gain_comparison.png"))

print("--- Figure 2: Layer-wise CKA Representational Divergence Curves ---")
display(Image("results/figures/cka_divergence_curve.png"))

print("--- Figure 3: TransferGuard Routing Profile on SVHN ---")
display(Image("results/figures/transferguard_svhn_routing_profile.png"))

print("--- Figure 4: Training Convergence Trajectories ---")
display(Image("results/figures/training_convergence_comparison.png"))

## 9. Run Interactive Live Terminal Demo

In [ ]:
!python scripts/demo_live.py

## 10. Package Figures & Checkpoints for Presentation

In [ ]:
!zip -r transferguard_presentation_results.zip results/figures/ results/benchmark_summary.csv docs/
from google.colab import files
files.download("transferguard_presentation_results.zip")